# Chapter 9, Exercise 4: A MUSHRA-like subjective evaluation lab for Arabic TTS

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 9, Exercise 4.** Design and deploy a hands-on subjective evaluation lab. Prepare 20 text-to-speech outputs (10 MSA, 10 dialectal), draft a naturalness annotation scheme, and deploy it on a platform such as Label Studio with a MUSHRA-style scale, a hidden reference, and low- and mid-quality anchors. Define the listener instructions, the bias-mitigation steps (randomization, native listeners, attention checks), and an objective WER cross-check, and state how you would report results per dialect with a confidence interval.

**Notes on what this notebook can and cannot do.** It generates the 20 stimuli with an open Arabic TTS model, builds the anchors and the hidden reference, writes a complete Label Studio project (task file plus labelling interface), runs the ASR-based WER cross-check, and shows the per-dialect confidence-interval computation on **simulated ratings that are clearly labelled as illustrative**. Real listener ratings must come from the deployed Label Studio project. Two data caveats: (1) `facebook/mms-tts-ara` is an MSA voice; no openly licensed dialectal Arabic TTS checkpoint with documented provenance could be verified for this release, so the ten "dialectal" stimuli are dialect *sentences* rendered by the MSA voice, which is itself a condition worth rating, and a slot is provided to plug in a dialect checkpoint; (2) the natural-speech references come from the FLEURS Arabic test split (MSA) and from a community-redistributed SADA subset (Saudi and Egyptian), whose licences you must verify before use in a published study.

## Requirements

Runs on the free CPU tier (MMS-TTS synthesis is a few seconds per sentence; Whisper-small on 20 short clips takes a few minutes). A GPU is optional.

## Testing status

Stimulus generation, anchor construction, Label Studio export, the WER cross-check and the CI computation (on simulated ratings) were executed. The listening test itself was not run; ratings in Section 6 are simulated and labelled as such.


In [1]:
!pip install -q "transformers>=4.40" torch soundfile scipy jiwer pandas huggingface_hub pyarrow librosa

## 1. Twenty sentences: 10 MSA, 10 dialectal (5 Saudi/Najdi, 5 Egyptian)

The sentences were written for this exercise (illustrative text). Each has an intended reading in ordinary spelling; the WER cross-check compares the recognizer's transcript with this text after normalization.

In [2]:
import pandas as pd
sentences = pd.DataFrame([
  ("msa01","MSA","أعلنت الوزارة عن افتتاح ثلاث مدارس جديدة في المنطقة الشرقية"),
  ("msa02","MSA","يبدأ العام الدراسي في الأسبوع الأول من شهر سبتمبر"),
  ("msa03","MSA","تشهد المدينة ارتفاعا ملحوظا في درجات الحرارة هذا الأسبوع"),
  ("msa04","MSA","أكد الباحثون أن النتائج تحتاج إلى مزيد من التحقق"),
  ("msa05","MSA","يمكن للمسافرين حجز التذاكر عبر التطبيق الجديد"),
  ("msa06","MSA","افتتح المعرض أبوابه أمام الزوار صباح اليوم"),
  ("msa07","MSA","تستغرق الرحلة من الرياض إلى جدة نحو ساعتين"),
  ("msa08","MSA","نظمت الجامعة ندوة حول الذكاء الاصطناعي واللغة العربية"),
  ("msa09","MSA","ارتفعت أسعار النفط في الأسواق العالمية أمس"),
  ("msa10","MSA","دعت البلدية السكان إلى ترشيد استهلاك المياه"),
  ("sau01","Saudi","وين أقرب محطة بنزين هنا"),
  ("sau02","Saudi","أبغى أروح السوق الحين بس الجو حار مرة"),
  ("sau03","Saudi","تعال نتغدى عندنا بكرة إن شاء الله"),
  ("sau04","Saudi","وش رايك نطلع البر نهاية الأسبوع"),
  ("sau05","Saudi","ما عندي وقت اليوم عندي اجتماع الساعة أربع"),
  ("egy01","Egyptian","أنا مش عارف أرد على كل الكومنتات دلوقتي"),
  ("egy02","Egyptian","إحنا رايحين فين النهارده"),
  ("egy03","Egyptian","الجو حر أوي النهارده مش قادر أخرج"),
  ("egy04","Egyptian","هقول لك وإحنا في الطريق"),
  ("egy05","Egyptian","عايز أشرب قهوة قبل ما نمشي"),
], columns=["id", "variety", "text"])
sentences["dialect_group"] = sentences["variety"].map({"MSA": "MSA", "Saudi": "dialectal", "Egyptian": "dialectal"})
sentences

,id,variety,text,dialect_group
0,msa01,MSA,أعلنت الوزارة عن افتتاح ثلاث مدارس جديدة في ال...,MSA
1,msa02,MSA,يبدأ العام الدراسي في الأسبوع الأول من شهر سبتمبر,MSA
2,msa03,MSA,تشهد المدينة ارتفاعا ملحوظا في درجات الحرارة ه...,MSA
3,msa04,MSA,أكد الباحثون أن النتائج تحتاج إلى مزيد من التحقق,MSA
4,msa05,MSA,يمكن للمسافرين حجز التذاكر عبر التطبيق الجديد,MSA
5,msa06,MSA,افتتح المعرض أبوابه أمام الزوار صباح اليوم,MSA
6,msa07,MSA,تستغرق الرحلة من الرياض إلى جدة نحو ساعتين,MSA
7,msa08,MSA,نظمت الجامعة ندوة حول الذكاء الاصطناعي واللغة ...,MSA
8,msa09,MSA,ارتفعت أسعار النفط في الأسواق العالمية أمس,MSA
9,msa10,MSA,دعت البلدية السكان إلى ترشيد استهلاك المياه,MSA


## 2. Synthesize the 20 stimuli

`facebook/mms-tts-ara` (VITS, 16 kHz, CC BY-NC 4.0) runs on CPU in a few seconds per sentence. It accepts undiacritized text and infers the vowels internally (Section 9.5), which is one of the things listeners will be judging. To use a dialect checkpoint, set `DIALECT_TTS` to its Hub id and adapt `synthesize()`.

In [3]:
import os, torch, numpy as np, soundfile as sf, warnings, transformers
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
from transformers import VitsModel, AutoTokenizer
os.makedirs("stimuli", exist_ok=True)
MSA_TTS = "facebook/mms-tts-ara"
DIALECT_TTS = None     # e.g. a documented dialect checkpoint; None = use the MSA voice for dialect text too

models = {}
def load(name):
    if name not in models:
        models[name] = (VitsModel.from_pretrained(name).eval(), AutoTokenizer.from_pretrained(name))
    return models[name]

def synthesize(text, variety):
    name = DIALECT_TTS if (variety != "MSA" and DIALECT_TTS) else MSA_TTS
    model, tok = load(name)
    with torch.no_grad():
        wav = model(**tok(text, return_tensors="pt")).waveform[0].numpy()
    return wav, model.config.sampling_rate, name

paths, voices = [], []
for _, r in sentences.iterrows():
    wav, sr, name = synthesize(r["text"], r["variety"])
    p = f"stimuli/{r['id']}_tts.wav"; sf.write(p, wav, sr); paths.append(p); voices.append(name)
sentences["tts_path"] = paths; sentences["tts_model"] = voices
print("synthesized", len(paths), "stimuli;", "models used:", set(voices))

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loading weights:  63%|██████▎   | 482/762 [00:00<00:00, 4703.10it/s]

Loading weights: 100%|██████████| 762/762 [00:00<00:00, 4264.69it/s]

synthesized 20 stimuli; models used: {'facebook/mms-tts-ara'}


## 3. Hidden reference and anchors (MUSHRA, ITU-R BS.1534)

* **Reference**: a natural recording. For each stimulus we attach one natural clip of the same variety (MSA from FLEURS; Saudi and Egyptian from the SADA subset). In a strict MUSHRA the reference is the *same sentence* recorded naturally; when no such recording exists, the test must be described as **MUSHRA-like** (Section 9.8).
* **Low anchor**: the reference low-pass filtered at 3.5 kHz; **mid anchor**: low-pass at 7 kHz (the standard MUSHRA anchors).
* The reference also appears **hidden** among the rated items, so a listener who rates it below 90 fails the attention check.

In [4]:
import io, pyarrow.parquet as pq, librosa
from huggingface_hub import hf_hub_download
from scipy.signal import butter, sosfiltfilt

def lowpass(y, sr, fc):
    return sosfiltfilt(butter(8, fc, btype="low", fs=sr, output="sos"), y).astype(np.float32)

# natural references
refs = {}
fl = hf_hub_download("google/fleurs", "parquet-data/ar_eg/test-00000-of-00001.parquet", repo_type="dataset")
batch = next(pq.ParquetFile(fl).iter_batches(batch_size=10)).to_pandas()
refs["MSA"] = [(sf.read(io.BytesIO(a["bytes"]))[0].astype(np.float32), 16000, t) for a, t in zip(batch["audio"], batch["raw_transcription"])]
sp = hf_hub_download("SarahUssama/sada-arabic-test-dataset-sample", "default/train/0000.parquet",
                     repo_type="dataset", revision="refs/convert/parquet")
pf = pq.ParquetFile(sp); tab = pf.read().to_pandas()
for variety, label in [("Saudi", "Saudi"), ("Egyptian", "Egyptian")]:
    sub = tab[(tab["SpeakerDialect"] == label) & (tab["Environment"].str.startswith("Clean")) & (tab["SegmentLength"].between(2, 6))].head(5)
    refs[variety] = []
    for _, r in sub.iterrows():
        y, sr = sf.read(io.BytesIO(r["audio"]["bytes"]))
        if y.ndim > 1: y = y.mean(axis=1)
        if sr != 16000: y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000); sr = 16000
        refs[variety].append((y.astype(np.float32), sr, r["GroundTruthText"]))

ref_paths, lo_paths, mid_paths = [], [], []
for k, r in sentences.iterrows():
    y, sr, _ = refs[r["variety"]][k % len(refs[r["variety"]])]
    y = y / (np.abs(y).max() + 1e-9) * 0.8
    base = f"stimuli/{r['id']}"
    sf.write(base + "_ref.wav", y, sr); sf.write(base + "_anchor35.wav", lowpass(y, sr, 3500), sr); sf.write(base + "_anchor70.wav", lowpass(y, sr, 7000), sr)
    ref_paths.append(base + "_ref.wav"); lo_paths.append(base + "_anchor35.wav"); mid_paths.append(base + "_anchor70.wav")
sentences["ref_path"], sentences["anchor_low"], sentences["anchor_mid"] = ref_paths, lo_paths, mid_paths
print("references and anchors written")

references and anchors written


## 4. The annotation scheme and the Label Studio project

**Attribute rated:** naturalness ("how much does this sound like a person speaking this variety naturally?"), on a **0 to 100 MUSHRA-style slider** with the labelled bands *Bad (0 to 20), Poor (20 to 40), Fair (40 to 60), Good (60 to 80), Excellent (80 to 100)*. Each screen presents the open reference plus four hidden items in random order: the TTS output, the hidden reference, the low anchor and the mid anchor. A second, separate question asks whether the **pronunciation** was correct (yes / one error / several errors) and a free-text box records mispronounced words, because naturalness and pronunciation are different attributes (Section 9.8).

**Listener instructions (shown once):** *You will hear a reference recording and four test recordings of Arabic speech. Use headphones in a quiet room. Listen to each recording fully, as often as you like. Rate how natural each one sounds compared with the reference, from 0 (completely unnatural) to 100 (indistinguishable from natural speech). At least one item should be rated 100 if it sounds identical to the reference. Rate only the speech quality, not the content or the topic. Note any word that is pronounced wrongly.*

**Bias mitigation, built into the export:** (1) item order and screen order are randomized per listener with a recorded seed; (2) listeners are recruited per variety and must self-identify as native speakers of it (MSA screens are rated by any native Arabic speaker; Saudi screens by Saudi listeners; Egyptian screens by Egyptian listeners), and a listener never sees the model identity; (3) **attention checks**: the hidden reference must be rated at or above 90 and the low anchor at or below the mid anchor on at least 80 % of screens, otherwise the listener's data are excluded; (4) a short training screen precedes the test; (5) each item is rated by at least 15 listeners; (6) playback device and listening environment are recorded.

In [5]:
import json, random
random.seed(20260916)
tasks = []
for _, r in sentences.iterrows():
    items = [("tts", r["tts_path"]), ("hidden_ref", r["ref_path"]), ("anchor_low", r["anchor_low"]), ("anchor_mid", r["anchor_mid"])]
    random.shuffle(items)                       # hidden items in random order; the label is kept only in the export key
    d = {"screen_id": r["id"], "variety": r["variety"], "reference": r["ref_path"], "text": r["text"]}
    for k, (kind, path) in enumerate(items, start=1):
        d[f"item{k}"] = path; d[f"item{k}_kind"] = kind     # kind is for analysis; hide it from the interface
    tasks.append({"data": d})
random.shuffle(tasks)                            # screen order randomized
json.dump(tasks, open("mushra_tasks.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

config = '''<View>
  <Header value="Screen $screen_id ($variety). Listen to the REFERENCE first."/>
  <Audio name="reference" value="$reference"/>
  <Header value="Rate the naturalness of each item from 0 (unnatural) to 100 (identical to natural speech)."/>
  <Audio name="a1" value="$item1"/><Rating name="r1" toName="a1" maxRating="100" icon="star" size="small"/>
  <Audio name="a2" value="$item2"/><Rating name="r2" toName="a2" maxRating="100" icon="star" size="small"/>
  <Audio name="a3" value="$item3"/><Rating name="r3" toName="a3" maxRating="100" icon="star" size="small"/>
  <Audio name="a4" value="$item4"/><Rating name="r4" toName="a4" maxRating="100" icon="star" size="small"/>
  <Header value="Pronunciation of the intended text: $text"/>
  <Choices name="pron" toName="a1" choice="single" showInline="true">
    <Choice value="all words correct"/><Choice value="one mispronounced word"/><Choice value="several mispronounced words"/>
  </Choices>
  <TextArea name="mispron" toName="a1" placeholder="list mispronounced words" rows="2"/>
  <Choices name="native" toName="reference" choice="single" showInline="true">
    <Choice value="I am a native speaker of this variety"/><Choice value="I am not"/>
  </Choices>
</View>'''
open("mushra_config.xml", "w").write(config)
print(len(tasks), "screens written to mushra_tasks.json; interface in mushra_config.xml")
print("Label Studio: New project > Labeling Interface > Code (paste XML) > Import mushra_tasks.json; serve the stimuli folder as local files.")

20 screens written to mushra_tasks.json; interface in mushra_config.xml
Label Studio: New project > Labeling Interface > Code (paste XML) > Import mushra_tasks.json; serve the stimuli folder as local files.


Label Studio's `Rating` widget is a star control; for a true slider install the Label Studio `Number`/range control or use webMUSHRA (Table 9.4), which implements the BS.1534 interface directly. Either way the exported ratings are numbers from 0 to 100 keyed by `screen_id` and item kind.

## 5. Objective cross-check: ASR-based WER on the synthesized stimuli

The intended text is the reference; the recognizer's transcript of the *synthesized* audio is the hypothesis. This is an intelligibility proxy only (Section 9.8, Table 9.3). The recognizer and the normalization must be named.

In [6]:
import re, unicodedata, jiwer
from transformers import pipeline
ASR = os.environ.get("ASR_MODEL", "openai/whisper-small")
asr = pipeline("automatic-speech-recognition", model=ASR, device=0 if torch.cuda.is_available() else -1,
               generate_kwargs={"language": "arabic", "task": "transcribe"})
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
def normalize(t):
    t = unicodedata.normalize("NFC", t).replace("\u0640", ""); t = DIAC.sub("", t)
    t = re.sub(r"[\u060C\u061B\u061F!-/:-@\[-`{-~]", " ", t)
    t = re.sub("[أإآٱ]", "ا", t).replace("ة", "ه").replace("ى", "ي")
    return re.sub(r"\s+", " ", t).strip()

hyps = []
for _, r in sentences.iterrows():
    y, sr = sf.read(r["tts_path"])
    if sr != 16000: y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000)
    hyps.append(asr({"raw": y.astype(np.float32), "sampling_rate": 16000})["text"].strip())
sentences["asr_hyp"] = hyps
rows = []
for grp, g in sentences.groupby("variety"):
    o = jiwer.process_words([normalize(t) for t in g["text"]], [normalize(h) for h in g["asr_hyp"]])
    rows.append({"variety": grp, "sentences": len(g), "N": sum(len(normalize(t).split()) for t in g["text"]),
                 "WER %": round(100*o.wer, 1), "CER %": round(100*jiwer.process_characters([normalize(t) for t in g["text"]], [normalize(h) for h in g["asr_hyp"]]).cer, 1)})
print("ASR:", ASR, "| normalization: diacritics/punct removed, alif/ta-marbuta/ya unified")
pd.DataFrame(rows)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  25%|██▍       | 118/479 [00:00<00:00, 1175.72it/s]

Loading weights:  49%|████▉     | 236/479 [00:00<00:00, 1094.98it/s]

Loading weights:  72%|███████▏  | 346/479 [00:00<00:00, 1082.75it/s]

Loading weights:  95%|█████████▍| 455/479 [00:00<00:00, 1062.67it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1094.16it/s]

ASR: openai/whisper-small | normalization: diacritics/punct removed, alif/ta-marbuta/ya unified


,variety,sentences,N,WER %,CER %
0,Egyptian,5,30,43.3,13.8
1,MSA,10,81,40.7,11.5
2,Saudi,5,34,23.5,9.1


In [7]:
sentences[["id", "variety", "text", "asr_hyp"]]

,id,variety,text,asr_hyp
0,msa01,MSA,أعلنت الوزارة عن افتتاح ثلاث مدارس جديدة في ال...,قلنات الوزارة عن افتتاح ثلاث مدارس جديدة في ال...
1,msa02,MSA,يبدأ العام الدراسي في الأسبوع الأول من شهر سبتمبر,يدأ العام الدراسي في الأسبوع الأولى من شهرين س...
2,msa03,MSA,تشهد المدينة ارتفاعا ملحوظا في درجات الحرارة ه...,أشهد المدينة رتفاعة ملحوظة في درجات الحرارة هذ...
3,msa04,MSA,أكد الباحثون أن النتائج تحتاج إلى مزيد من التحقق,أكل بحثون أن النتائج تحتاج إلى مزيد من التحقق
4,msa05,MSA,يمكن للمسافرين حجز التذاكر عبر التطبيق الجديد,يوكنوا للمسافرين حجز تذاكرة عبر التطبيق الجديد
5,msa06,MSA,افتتح المعرض أبوابه أمام الزوار صباح اليوم,هتتح المعرض أبوابه أمام الزوان الصباح اليون
6,msa07,MSA,تستغرق الرحلة من الرياض إلى جدة نحو ساعتين,كاستغرق الرحلة من الرياضي إلى جدة نحسعتي
7,msa08,MSA,نظمت الجامعة ندوة حول الذكاء الاصطناعي واللغة ...,لما تلجامعة الندوة حول الذكاء الله صطناعي ولغة...
8,msa09,MSA,ارتفعت أسعار النفط في الأسواق العالمية أمس,ارتفعت أصعار النفط في الأسواق العالمية أمسا
9,msa10,MSA,دعت البلدية السكان إلى ترشيد استهلاك المياه,ذات البلدية السكان إلى ترشيد استهلاك المياه


## 6. Reporting per dialect with a confidence interval

For each variety and each item kind, compute the mean rating over listeners and screens and a 95 % confidence interval. Because listeners rate several screens, the correct unit of replication is the **listener**: average each listener's ratings per variety first, then compute the CI over listener means (a t-interval, or a bootstrap over listeners). The ratings below are **simulated** to show the computation; replace `ratings` with the Label Studio export.

In [8]:
from scipy import stats
rng = np.random.default_rng(0)
# SIMULATED ratings: 15 listeners x 20 screens x 4 items. Illustrative only.
sim = []
for listener in range(15):
    for t in tasks:
        v = t["data"]["variety"]
        for k in range(1, 5):
            kind = t["data"][f"item{k}_kind"]
            mu = {"hidden_ref": 96, "anchor_mid": 70, "anchor_low": 35}.get(kind, {"MSA": 62, "Saudi": 45, "Egyptian": 43}[v])
            sim.append({"listener": listener, "screen": t["data"]["screen_id"], "variety": v, "kind": kind,
                        "rating": float(np.clip(rng.normal(mu, 10), 0, 100))})
ratings = pd.DataFrame(sim)

def ci_over_listeners(df):
    per_listener = df.groupby("listener")["rating"].mean()
    m, se = per_listener.mean(), per_listener.std(ddof=1)/np.sqrt(len(per_listener))
    h = stats.t.ppf(0.975, len(per_listener)-1) * se
    return pd.Series({"mean": m, "CI95 low": m-h, "CI95 high": m+h, "n listeners": len(per_listener)})

report = ratings.groupby(["variety", "kind"]).apply(ci_over_listeners).round(1)
print("SIMULATED ratings, illustrative only")
report

SIMULATED ratings, illustrative only


mean  CI95 low  CI95 high  n listeners
variety  kind                                              
Egyptian anchor_low  36.1      33.7       38.6         15.0
         anchor_mid  71.1      68.6       73.6         15.0
         hidden_ref  94.0      92.3       95.6         15.0
         tts         40.6      38.4       42.7         15.0
MSA      anchor_low  33.7      32.0       35.3         15.0
         anchor_mid  70.2      68.8       71.7         15.0
         hidden_ref  93.1      92.2       94.0         15.0
         tts         62.0      60.1       63.8         15.0
Saudi    anchor_low  34.7      32.8       36.6         15.0
         anchor_mid  70.2      67.5       72.8         15.0
         hidden_ref  93.0      91.0       95.1         15.0
         tts         44.7      42.0       47.5         15.0

**How to report.** One row per variety and item kind (as above), with the mean, the 95 % CI over listeners, the number of listeners and the number of screens; the hidden-reference and anchor rows are reported too, because they validate the test (the reference should be near 100, the anchors ordered). State the MUSHRA-like design, the exact question and scale, the listener recruitment and native-speaker screening, the attention-check exclusions, the playback conditions, the TTS model and version for each variety (here the MSA voice for all, which must be said), and the ASR model and normalization behind the WER cross-check. Never pool MSA and dialectal screens into one naturalness number, and when the dialectal stimuli were produced by an MSA voice, report that condition as what it is: an MSA voice reading dialect text, rated by native listeners of that dialect.